In [50]:
%%capture
%load_ext autoreload
%autoreload 2

In [51]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.algorithms.recommenders import ItemBasedCFRecommender
from IPython.display import display
from recsys_pipeliner.evaluation import (
    AccuracyMetrics,
    AlgorithmEvaluator,
    EvaluationDataset,
    TopNMetrics,
)
from IPython.display import display

In [52]:
# load test data
ratings_data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings_df = pd.read_csv(
    "../../tests/test_data/user_item_ratings_toy.csv", dtype=ratings_data_types
)
item_features_df = pd.read_csv(
    "../../tests/test_data/item_features_toy.csv", index_col=0
)

display(user_item_ratings_df.head(3))
display(item_features_df.head(3))

,user_id,item_id,rating
0,U00001,I00024,0.8
1,U00001,I00013,0.6
2,U00001,I00005,1.0


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10
I00001,0.65,0.71,0.75,0.91,0.70,0.75,0.88,0.95,0.62,0.87
I00002,0.57,0.56,0.14,0.33,0.39,0.29,0.17,0.13,0.11,0.60
I00003,0.46,0.65,0.51,0.69,0.47,0.49,0.49,0.59,0.68,0.72


In [53]:
# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings_df["item_id"] = item_encoder.fit_transform(user_item_ratings_df["item_id"])
user_item_ratings_df["user_id"] = user_encoder.fit_transform(user_item_ratings_df["user_id"])

user_item_ratings_np = user_item_ratings_df.to_numpy()

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

print("unique_users", unique_users.shape[0])
print("unique_items", unique_items.shape[0])

unique_users 12
unique_items 24


In [54]:
dataset = EvaluationDataset(
    user_item_ratings_np, 
    min_user_ratings=2, 
    min_item_ratings=2, 
    random_seed=42
)
trainset, testset = dataset.trainset, dataset.testset
anti_testset = dataset.anti_testset
usable_ratings = dataset.usable

print("user_item_ratings_np", user_item_ratings_np.shape)
print("trainset", trainset.shape)
print("testset", testset.shape)
print("anti_testset", anti_testset.shape)
print("usable_ratings", usable_ratings.shape)

assert set(np.unique(trainset[:, 0]).astype(int)) == set(unique_users.index)
assert (
    set(np.unique(testset[:, 0]).astype(int)) == set(unique_users.index)
)

user_item_ratings_np (96, 3)
trainset (84, 3)
testset (12, 3)
anti_testset (192, 2)
usable_ratings (96, 3)


In [55]:
# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    trainset,
)
print("user_item_matrix", user_item_matrix.shape)

# sanity check
users = trainset[:, 0].astype(int)
items = trainset[:, 1].astype(int)
ratings = trainset[:, 2].astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

user_item_matrix (12, 24)


In [56]:
users = anti_testset[:, 0]
items = anti_testset[:, 1]

# sanity check
for user, item in anti_testset:
    assert user_item_matrix[user, item] == 0

In [57]:
rec = ItemBasedCFRecommender(k=5, n=5)
evaluator = AlgorithmEvaluator(rec)

result = evaluator.evaluate(
    user_item_matrix, dataset.testset, dataset.anti_testset, top_n=5
)

result

(AccuracyMetrics(rmse=0.3305, mae=0.2694),
 TopNMetrics(HR=0.25, cHR=0.25, ARHR=0.083333))

In [ ]:
# sense check results

# TODO: this seems to be wrong? 
# Need to go back and confirm expected recommendations and predictions manually.
# Not sure if the unit tests validation data is correct.

rec2 = ItemBasedCFRecommender(k=5, n=5)
rec2.fit(user_item_matrix)
test_input = dataset.testset[:, 0:2].astype(np.int32)
test_predictions = dataset.testset[:, 2]

recommendations = rec2.recommend(test_input)

for u, rec in zip(test_input[:, 0], recommendations):
    us = np.repeat(u, len(rec))
    pred_input = np.column_stack((us, rec))
    prediction = rec2.predict(pred_input)
    results = np.column_stack((pred_input, prediction))
    print(results)

[[ 0.          8.          0.2       ]
 [ 0.         18.          0.46693999]
 [ 0.         16.          0.235594  ]
 [ 0.         21.          0.44581199]
 [ 0.          1.          0.59200698]]
[[ 1.         10.          0.72965997]
 [ 1.          5.          0.72856498]
 [ 1.          3.          0.64856899]
 [ 1.         20.          0.69782197]
 [ 1.         22.          0.70173001]]
[[ 2.          4.          0.92676699]
 [ 2.          0.          0.971084  ]
 [ 2.          6.          0.74086702]
 [ 2.         20.          0.86920202]
 [ 2.         12.          0.739851  ]]
[[ 3.         19.          0.63503599]
 [ 3.         22.          0.56914502]
 [ 3.          4.          0.65936399]
 [ 3.          9.          0.68295401]
 [ 3.         18.          0.558806  ]]
[[ 4.          6.          0.78526503]
 [ 4.          2.          0.72854197]
 [ 4.         10.          0.742688  ]
 [ 4.         22.          0.489889  ]
 [ 4.         23.          0.68590999]]
[[ 5.         16.   

# Hybrid Recommender

This recommender uses item features to calcuate similarity and user/item ratings to rank recommendations.

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

# Content-based filtering and a hybrid recommender

item_features_np = item_features_df.to_numpy().astype(np.float32)

# NOTE: this is the only difference to how the ItemBasedCFRecommender is implemented.
# We just use the item features to calculate item similarity.
item_similarity_matrix = cosine_similarity(item_features_np).astype(np.float32).round(6)



predictions = []

for user_idx, item_idx in anti_testset:
    _, target_user_rated_items, target_user_ratings = sp.sparse.find(
        user_item_matrix[user_idx, :]
    )

    # exclude item_idx if already rated by target user
    target_user_rated_items = target_user_rated_items[target_user_rated_items != item_idx]
    users_ratings = target_user_ratings[target_user_rated_items != item_idx]

    # get the item similarities to item_idx
    item_similarities = (
        item_similarity_matrix[:, target_user_rated_items][item_idx]
        .astype(np.float32)
        .round(6)
    )
    k = 5
    # sort by similarity (desc) and get top k
    top_k_mask = np.argsort(1 - item_similarities)[:k]
    top_k_target_user_ratings = users_ratings[top_k_mask]
    top_k_rated_item_similarities = item_similarities[top_k_mask]
    # weighted average rating
    prediction = (
        np.average(top_k_target_user_ratings, axis=0, weights=top_k_rated_item_similarities)
        .astype(np.float32)
        .round(6)
    )
    predictions.append([user_idx, item_idx, prediction])

predictions

[[0, 0, 0.63807],
 [0, 1, 0.638394],
 [0, 2, 0.63852],
 [0, 3, 0.64014],
 [0, 5, 0.482618],
 [0, 6, 0.640523],
 [0, 7, 0.602654],
 [0, 8, 0.637624],
 [0, 9, 0.640147],
 [0, 13, 0.595282],
 [0, 15, 0.638849],
 [0, 16, 0.636553],
 [0, 18, 0.640008],
 [0, 19, 0.717574],
 [0, 20, 0.640535],
 [0, 21, 0.640334],
 [1, 0, 0.760667],
 [1, 1, 0.765463],
 [1, 3, 0.763056],
 [1, 4, 0.761822],
 [1, 5, 0.728877],
 [1, 6, 0.762332],
 [1, 7, 0.732235],
 [1, 8, 0.759856],
 [1, 9, 0.761205],
 [1, 10, 0.761991],
 [1, 12, 0.762343],
 [1, 15, 0.761501],
 [1, 19, 0.76472],
 [1, 20, 0.761425],
 [1, 22, 0.76287],
 [1, 23, 0.761576],
 [2, 0, 0.851657],
 [2, 1, 0.852841],
 [2, 4, 0.851584],
 [2, 6, 0.85222],
 [2, 7, 0.852086],
 [2, 8, 0.851606],
 [2, 9, 0.8521],
 [2, 11, 0.853226],
 [2, 12, 0.851334],
 [2, 14, 0.851448],
 [2, 16, 0.850871],
 [2, 17, 0.853542],
 [2, 19, 0.849565],
 [2, 20, 0.851885],
 [2, 21, 0.852543],
 [2, 22, 0.85119],
 [3, 1, 0.585239],
 [3, 2, 0.584808],
 [3, 4, 0.583762],
 [3, 5, 0.583467]